# Shallow LSTM Benchmark with Optuna & Symlog

In [13]:
!pip install optuna

In [14]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler, RobustScaler
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pack_padded_sequence
from torch.optim.lr_scheduler import ReduceLROnPlateau
import optuna

SEED = 42
DATA_PATH = Path("Merged_Dataset_yoy.csv")
PREDICTION_TARGETS = ["EBITDA", "Net_Income", "ROA"]
DEFAULT_LOOKBACK_DAYS = 365

MAX_EPOCHS = 100
PATIENCE = 10
TRAIN_YEAR_CUTOFF = 2019
VALID_YEAR_CUTOFF = 2021
OPTUNA_TRIALS = 100

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(max(1, torch.get_num_threads() // 2))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


## Data Processing (Baseline)

In [15]:
def load_and_prepare_yoy_data(data_path: Path, lookback_days: int = 365):
    try:
        df = pd.read_csv(data_path)
    except FileNotFoundError:
        df = pd.read_csv("Merged_Dataset_yoy.csv")

    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values(["Company", "Date"]).reset_index(drop=True)
    df["year"] = df["Date"].dt.year

    df["has_targets"] = df[PREDICTION_TARGETS].notna().any(axis=1)
    year_end_records = df[df["has_targets"]].copy()
    exclude_cols = {"Date", "Company", "year", "has_targets"} | set(PREDICTION_TARGETS)
    feature_cols = [col for col in df.columns if col not in exclude_cols and pd.api.types.is_numeric_dtype(df[col])]

    sequences = []
    for company in df["Company"].unique():
        company_records = year_end_records[year_end_records["Company"] == company].copy()
        company_data = df[df["Company"] == company].copy()

        for _, year_end_row in company_records.iterrows():
            year = int(year_end_row["year"])
            year_end_date = year_end_row["Date"]

            prior_year_records = year_end_records[
                (year_end_records["Company"] == company) & (year_end_records["year"] == year - 1)
            ]
            if prior_year_records.empty:
                continue

            prior_row = prior_year_records.iloc[0]
            window_start = year_end_date - pd.Timedelta(days=lookback_days)
            window = company_data[(company_data["Date"] > window_start) & (company_data["Date"] <= year_end_date)].copy()

            if len(window) < 200:
                continue

            window[feature_cols] = window[feature_cols].ffill().bfill()
            if window[feature_cols].isna().any().any():
                continue

            sequences.append({
                "company": company, "year": year, "year_end_date": year_end_date,
                "window_data": window[feature_cols].to_numpy(dtype=np.float32),
                **{f"current_{t.lower()}": year_end_row[t] for t in PREDICTION_TARGETS},
                **{f"prior_{t.lower()}": prior_row[t] for t in PREDICTION_TARGETS},
            })

    data_records = []
    for seq in sequences:
        for target_name in PREDICTION_TARGETS:
            current_val = seq[f"current_{target_name.lower()}"]
            prior_val = seq[f"prior_{target_name.lower()}"]
            if pd.isna(current_val) or pd.isna(prior_val):
                continue

            label_value = (current_val - prior_val) / (np.abs(prior_val) + 1e-8)

            data_records.append({
                "company": seq["company"], "year": seq["year"], "year_end_date": seq["year_end_date"],
                "target": target_name, "label_value": label_value,
                "window_data": seq["window_data"]
            })

    return pd.DataFrame(data_records), feature_cols

data_df, feature_cols = load_and_prepare_yoy_data(DATA_PATH, lookback_days=DEFAULT_LOOKBACK_DAYS)

def split_by_year(data_df: pd.DataFrame, train_cutoff: int, valid_cutoff: int):
    train_data = data_df[data_df["year"] <= train_cutoff].copy()
    val_data = data_df[(data_df["year"] > train_cutoff) & (data_df["year"] <= valid_cutoff)].copy()
    test_data = data_df[data_df["year"] > valid_cutoff].copy()
    return train_data, val_data, test_data

train_data, val_data, test_data = split_by_year(data_df, TRAIN_YEAR_CUTOFF, VALID_YEAR_CUTOFF)

scaler = StandardScaler()
train_windows = np.vstack([row for row in train_data["window_data"]])
scaler.fit(train_windows)

max_seq_length = max(len(row) for row in data_df["window_data"])


## Enhanced Pipeline Setup (Symlog, RobustScaler, Packing)

In [16]:
SELECTED_FEATURES = feature_cols.copy() # Can apply feature selection here
selected_indices = [feature_cols.index(f) for f in SELECTED_FEATURES]

improved_data_df = data_df.copy()

# SYMLOG TRANSFORMATION instead of Winsorization
idx_value = improved_data_df["target"].isin(PREDICTION_TARGETS)
y_val = improved_data_df.loc[idx_value, "label_value"]
improved_data_df.loc[idx_value, "label_value"] = np.sign(y_val) * np.log1p(np.abs(y_val))

train_data_imp, val_data_imp, test_data_imp = split_by_year(improved_data_df, TRAIN_YEAR_CUTOFF, VALID_YEAR_CUTOFF)

robust_scaler = RobustScaler()
train_windows_imp = np.vstack([row[:, selected_indices] for row in train_data_imp["window_data"]])
robust_scaler.fit(train_windows_imp)

def collate_fn_improved(batch):
    batch.sort(key=lambda x: x[2], reverse=True)
    sequences = [x[0] for x in batch]
    targets = torch.stack([x[1] for x in batch])
    lengths = torch.tensor([x[2] for x in batch])
    padded_seqs = torch.nn.utils.rnn.pad_sequence(sequences, batch_first=True)
    return padded_seqs, targets, lengths


## Model Definitions

In [17]:
class ShallowLSTMClassifier(nn.Module):
    def __init__(self, input_size: int, hidden_size: int, num_targets: int = 1, dropout: float = 0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=1, batch_first=True)
        self.dropout = nn.Dropout(p=dropout)
        self.head = nn.Linear(hidden_size, num_targets)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        lstm_out, _ = self.lstm(x)
        hidden = self.dropout(lstm_out[:, -1, :])
        logits = self.head(hidden)
        if logits.size(-1) == 1: logits = logits.squeeze(-1)
        return logits

class ImprovedShallowLSTM(nn.Module):
    def __init__(self, input_size: int, hidden_size: int, num_targets: int = 1, dropout: float = 0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=1, batch_first=True)
        self.dropout = nn.Dropout(p=dropout)
        self.head = nn.Linear(hidden_size, num_targets)

    def forward(self, x, lengths):
        packed_x = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=True)
        _, (hn, _) = self.lstm(packed_x)
        hidden = self.dropout(hn[-1])
        logits = self.head(hidden)
        if logits.size(-1) == 1: logits = logits.squeeze(-1)
        return logits


## Helpers for Optuna and Evaluation

In [18]:
def extract_mtl_df(df: pd.DataFrame) -> pd.DataFrame:
    mtl_records = []
    for (company, year), group in df.groupby(["company", "year"]):
        window_data = group.iloc[0]["window_data"]
        year_end_date = group.iloc[0]["year_end_date"]
        label_value = np.full(len(PREDICTION_TARGETS), np.nan, dtype=np.float32)
        for i, t in enumerate(PREDICTION_TARGETS):
            t_row = group[group["target"] == t]
            if not t_row.empty:
                label_value[i] = t_row.iloc[0]["label_value"]
        mtl_records.append({
            "company": company, "year": year, "year_end_date": year_end_date,
            "window_data": window_data, "label_value": label_value
        })
    return pd.DataFrame(mtl_records)

# Baseline MTL dfs
mtl_train_data = extract_mtl_df(train_data)
mtl_val_data = extract_mtl_df(val_data)
mtl_test_data = extract_mtl_df(test_data)

# Improved MTL dfs
mtl_train_data_imp = extract_mtl_df(train_data_imp)
mtl_val_data_imp = extract_mtl_df(val_data_imp)
mtl_test_data_imp = extract_mtl_df(test_data_imp)

class YoYSequenceDatasetMTL(Dataset):
    def __init__(self, data_df: pd.DataFrame, max_seq_length: int, scaler: StandardScaler = None):
        self.data_df = data_df.reset_index(drop=True)
        self.max_seq_length = max_seq_length
        self.scaler = scaler

    def __len__(self) -> int: return len(self.data_df)

    def __getitem__(self, idx: int):
        row = self.data_df.iloc[idx]
        window = row["window_data"].copy()
        if self.scaler is not None: window = self.scaler.transform(window)
        seq_len = len(window)
        if seq_len < self.max_seq_length:
            padding = np.zeros((self.max_seq_length - seq_len, window.shape[1]), dtype=np.float32)
            window = np.vstack([padding, window])
        target = row["label_value"]
        mask = ~np.isnan(target)
        target_clean = np.nan_to_num(target, nan=0.0)
        return torch.from_numpy(window), torch.tensor(target_clean, dtype=torch.float32), torch.tensor(mask, dtype=torch.bool)


class ImprovedYoYDatasetMTL(Dataset):
    def __init__(self, data_df, scaler, feature_indices):
        self.data_df = data_df.reset_index(drop=True)
        self.scaler = scaler
        self.feature_indices = feature_indices

    def __len__(self): return len(self.data_df)

    def __getitem__(self, idx):
        row = self.data_df.iloc[idx]
        window = row["window_data"][:, self.feature_indices].copy()
        if self.scaler: window = self.scaler.transform(window)
        target = row["label_value"]
        mask = ~np.isnan(target)
        target_clean = np.nan_to_num(target, nan=0.0)
        return torch.from_numpy(window), torch.tensor(target_clean, dtype=torch.float32), torch.tensor(mask, dtype=torch.bool), len(window)

def collate_fn_improved_mtl(batch):
    batch.sort(key=lambda x: x[3], reverse=True)
    sequences = [x[0] for x in batch]
    targets = torch.stack([x[1] for x in batch])
    masks = torch.stack([x[2] for x in batch])
    lengths = torch.tensor([x[3] for x in batch])
    padded_seqs = torch.nn.utils.rnn.pad_sequence(sequences, batch_first=True)
    return padded_seqs, targets, masks, lengths


In [19]:
def build_loaders_and_model(is_improved, params):
    dropout_val = params.get("dropout", 0.2)
    if is_improved:
        train_ds = ImprovedYoYDatasetMTL(mtl_train_data_imp, robust_scaler, selected_indices)
        val_ds = ImprovedYoYDatasetMTL(mtl_val_data_imp, robust_scaler, selected_indices)
        test_ds = ImprovedYoYDatasetMTL(mtl_test_data_imp, robust_scaler, selected_indices)
        collate = collate_fn_improved_mtl
        model = ImprovedShallowLSTM(len(selected_indices), params["hidden_size"], len(PREDICTION_TARGETS), dropout_val).to(DEVICE)
    else:
        train_ds = YoYSequenceDatasetMTL(mtl_train_data, max_seq_length, scaler)
        val_ds = YoYSequenceDatasetMTL(mtl_val_data, max_seq_length, scaler)
        test_ds = YoYSequenceDatasetMTL(mtl_test_data, max_seq_length, scaler)
        collate = None
        model = ShallowLSTMClassifier(len(feature_cols), params["hidden_size"], len(PREDICTION_TARGETS), dropout_val).to(DEVICE)

    if collate:
        train_loader = DataLoader(train_ds, batch_size=params["batch_size"], shuffle=True, collate_fn=collate)
        val_loader = DataLoader(val_ds, batch_size=params["batch_size"], shuffle=False, collate_fn=collate)
        test_loader = DataLoader(test_ds, batch_size=params["batch_size"], shuffle=False, collate_fn=collate)
    else:
        train_loader = DataLoader(train_ds, batch_size=params["batch_size"], shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=params["batch_size"], shuffle=False)
        test_loader = DataLoader(test_ds, batch_size=params["batch_size"], shuffle=False)

    return model, train_loader, val_loader, test_loader


def evaluate_loader(model, loader, is_improved):
    model.eval()
    all_preds, all_targets, all_masks = [], [], []
    with torch.no_grad():
        for batch in loader:
            if is_improved:
                bx, by, bmask, lengths = batch
                logits = model(bx.to(DEVICE), lengths)
                all_masks.append(bmask.numpy())
            else:
                bx, by, bmask = batch
                logits = model(bx.to(DEVICE))
                all_masks.append(bmask.numpy())

            all_preds.append(logits.cpu().numpy())
            all_targets.append(by.numpy())

    if not all_preds:
        return None

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_targets)

    if is_improved:
        y_pred = np.sign(y_pred) * np.expm1(np.abs(y_pred))
        y_true = np.sign(y_true) * np.expm1(np.abs(y_true))

    mask = np.vstack(all_masks)
    res = {}
    for i, t in enumerate(PREDICTION_TARGETS):
        m = mask[:, i]
        if m.sum() == 0:
            continue
        res[t] = np.mean(np.abs(y_pred[m, i] - y_true[m, i]))
    return res


def final_train_and_test(is_improved, params):
    model, train_loader, val_loader, test_loader = build_loaders_and_model(
        is_improved, params
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])
    huber_delta = params.get("huber_delta", 1.0)

    model, _ = train_and_eval(
        model, train_loader, val_loader, optimizer,
        is_improved=is_improved, huber_delta=huber_delta
    )

    return evaluate_loader(model, test_loader, is_improved=is_improved)

def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)

N_RUNS = 20
print(f"Computing Final Test Metrics using Test Sets across {N_RUNS} runs...")
final_results = []

def run_multiple_times(pipeline_name, is_improved, best_params_dict):
    if "value" not in best_params_dict:
        return
    params = best_params_dict["value"]

    target_results = {t: [] for t in PREDICTION_TARGETS}
    for run_idx in range(N_RUNS):
        set_seed(SEED + run_idx)
        res = final_train_and_test(is_improved, params)
        for target, val in res.items():
            target_results[target].append(val)

    for target in PREDICTION_TARGETS:
        avg_mae = np.mean(target_results[target])
        std_mae = np.std(target_results[target])
        final_results.append({
            "Pipeline": pipeline_name,
            "Target": target,
            "Test MAE (Mean)": avg_mae,
            "Deviation": std_mae
        })



Computing Final Test Metrics using Test Sets across 20 runs...


In [20]:
mib_tickers = [
    "A2A", "AMPLIFON", "AZIMUT", "BANCA MEDIOLANUM", "BANCA MONTE DEI PASCHI",
    "BANCA PPO.DI SONDRIO", "BANCO BPM", "BPER BANCA", "BRUNELLO CUCINELLI",
    "BUZZI", "DAVIDE CAMPARI MILANO", "DIASORIN", "ENEL", "ENI",
    "FERRARI (MIL)", "FINCANTIERI", "FINECOBANK", "GENERALI", "HERA",
    "INFRASTRUTTURE WIRELESS ITALIANE NPV", "INTESANPAOLO", "ITALGAS",
    "IVECO", "LEONARDO", "LOTTOMATICA", "MEDIOBANCA BC.FIN", "MONCLER",
    "NEXI", "POSTE ITALIANE", "PRYSMIAN", "RECORDATI INDUA.CHIMICA",
    "SAIPEM", "SNAM", "STELLANTIS", "STMICROELECTRONICS (MIL)",
    "TELECOM ITALIA", "TENARIS", "TERNA RETE ELETTRICA NAZ", "UNICREDIT",
    "UNIPOL ASSICURAZIONI"
]

mid_tickers = [
    "ACEA", "ALERION CLEAN POWER", "ANIMA", "ARISTON",
    "ARNOLDO MONDADORI EDI.", "ASCOPIAVE", "AVIO", "BANCA GENERALI",
    "BANCA IFIS", "BFF BANK", "BNC.DI DESIO E DELB.", "CALTAGIRONE",
    "CAREL", "CEMBRE", "CEMENTIR", "COMPAGNIE INDUSTRIALI RIUNITE SHS",
    "CREDITO EMILIANO", "D'AMICO INTL.SHIP.", "DANIELI", "DATALOGIC",
    "DE LONGHI", "EL EN", "ENAV", "ERG", "FERRETTI (MIL)",
    "FIERA MILANO", "FRENI BREMBO", "GVS", "INTERCOS", "INTERPUMP",
    "IREN", "ITALMOBILIARE", "JUVENTUS FOOTBALL CLUB", "LUVE", "MARIE",
    "MARR", "MFE B", "MFE-MEDIAFOREUROPE", "MOLTIPLY", "NEWPRINCES",
    "OVS", "PHARMANUTRA", "PHILOGEN", "PIRELLI & C", "RAI WAY",
    "REPLY", "RIZZOLI CRER.DLSM.GP.", "SAFILO", "SALVATORE FERRAGAMO",
    "SANLORENZO", "SESA", "SOL", "TECHNOGYM", "TECHNOPROBE", "TINEXTA",
    "WEBUILD", "WIIT", "ZIGNAGO VETRO"
]

small_tickers = [
    "ABITARE IN", "AEDES", "AEFFE", "AEROP GUGL MARCO",
    "ALTEA GREEN POWER", "ANTARES VISION", "AQUAFIL", "B&C SPEAKERS",
    "BANCA PROFILO", "BANCA SISTEMA", "BASICNET", "BASTOGI", "BEEWIZE",
    "BIESSE", "BORGOSESIA", "BRIOSCHI SVILUPPO IMMBL", "CAIRO COMMUNICATION",
    "CALEFFI", "CALTAGIRONE EDITORE", "CELLULARLINE",
    "CENTRALE DEL LATTE D'ITALIA", "CLASS EDITORI", "CSP INTERNATIONAL",
    "CY4GATE", "DIGITAL BROS", "DIGITAL VALUE", "DOVALUE", "ELICA", "EMAK",
    "ENERVIT", "EPRICE", "EQUITA", "ESPRINET", "EUROGROUP LAMINATIONS A",
    "EUROTECH", "FIDIA", "FILA", "FNM", "GABETTI PROPERTY SLTN.",
    "GAROFALO HEALTH CARE", "GAS PLUS", "GEFRAN", "GENERALFINANCE",
    "GEOX", "GPI", "GRANDI VIAGGI", "IMMOBILIARE GRDE. DTBZ. SO.DI INVM.IMMB.",
    "IMMSI", "INDEL B", "INDUSTRIE DE NORA", "IRCE", "IT WAY",
    "ITALIAN DESIGN BRAND", "ITALIAN EXHIBITION", "ITALIAN SEA",
    "LANDI RENZO", "MET EXTRA", "MONDO TV", "NEODECORTECH", "OLIDATA",
    "OPS ITALIA", "OPS RETAIL", "ORSERO", "PININFARINA", "PIQUADRO",
    "PLC", "RATTI", "REVO INSURANCE", "RISANAMENTO", "SABAF", "SECO",
    "SERI INDUSTRIAL", "SIT", "SOFTLAB", "SOGEFI", "SOMEC", "SS LAZIO",
    "SYS-DAT", "TESMEC", "TESSELLIS", "TREVI FIN INDUSTRIALE", "TRIBOO",
    "TXT E-SOLUTION", "UNIDATA", "VALSOIA", "VINCENZO ZUCCHI", "ZEST"
]

ticker_groups = {"MIB": mib_tickers, "Mid-cap": mid_tickers, "Small-cap": small_tickers}

## Core Training Logic & Optuna Integration

In [21]:
def train_and_eval(model, train_loader, val_loader, optimizer, is_improved, trial=None, loss_type='l1', huber_delta=1.0):
    if loss_type == 'l1':
        criterion = nn.L1Loss(reduction='none')
    elif loss_type == 'mse':
        criterion = nn.MSELoss(reduction='none')
    elif loss_type == 'huber':
        criterion = nn.HuberLoss(delta=huber_delta, reduction='none')
    else:
        raise ValueError(f"Unknown loss_type: {loss_type}")

    scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

    best_state = None
    best_metric = float("inf")
    epochs_without_improvement = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        for batch in train_loader:
            optimizer.zero_grad(set_to_none=True)
            if is_improved:
                bx, by, bmask, lengths = batch
                bx, by, bmask = bx.to(DEVICE), by.to(DEVICE), bmask.to(DEVICE)
                logits = model(bx, lengths)
                loss_m = criterion(logits, by)
                loss = loss_m[bmask].mean() if loss_m[bmask].numel() > 0 else 0*loss_m.sum()
            else:
                bx, by, bmask = batch
                bx, by, bmask = bx.to(DEVICE), by.to(DEVICE), bmask.to(DEVICE)
                logits = model(bx)
                loss_m = criterion(logits, by)
                loss = loss_m[bmask].mean() if loss_m[bmask].numel() > 0 else 0*loss_m.sum()

            if isinstance(loss, torch.Tensor) and loss.requires_grad:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

        # Eval
        model.eval()
        all_preds = []
        all_targets = []
        all_masks = []
        with torch.no_grad():
            for batch in val_loader:
                if is_improved:
                    bx, by, bmask, lengths = batch
                    bx = bx.to(DEVICE)
                    logits = model(bx, lengths)
                    all_masks.append(bmask.numpy())
                else:
                    bx, by, bmask = batch
                    bx = bx.to(DEVICE)
                    logits = model(bx)
                    all_masks.append(bmask.numpy())

                all_preds.append(logits.cpu().numpy())
                all_targets.append(by.numpy())

        if len(all_preds) == 0:
            break

        y_pred = np.vstack(all_preds)
        y_true = np.vstack(all_targets)

        if is_improved:
            y_pred = np.sign(y_pred) * np.expm1(np.abs(y_pred))
            y_true = np.sign(y_true) * np.expm1(np.abs(y_true))

        mask = np.vstack(all_masks)
        valid_sum, count = 0, 0
        for i in range(len(PREDICTION_TARGETS)):
            m = mask[:, i]
            if m.sum() == 0: continue
            valid_sum += np.mean(np.abs(y_pred[m, i] - y_true[m, i]))
            count += 1
        current_metric = valid_sum / max(1, count)

        scheduler.step(current_metric)

        improved = current_metric < best_metric
        if improved:
            best_metric = current_metric
            best_state = {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if trial is not None:
            trial.report(current_metric, epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

        if epochs_without_improvement >= PATIENCE:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_metric

def get_hpo_objective(is_improved, loss_type='l1'):
    def objective(trial):
        hidden_size = trial.suggest_categorical("hidden_size", [32, 64, 128])
        lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
        batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
        weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
        dropout = trial.suggest_float("dropout", 0.2, 0.5)
        huber_delta = trial.suggest_float("huber_delta", 0.8, 2.0) if loss_type == 'huber' else 1.0


        if is_improved:
            train_ds = ImprovedYoYDatasetMTL(mtl_train_data_imp, robust_scaler, selected_indices)
            val_ds = ImprovedYoYDatasetMTL(mtl_val_data_imp, robust_scaler, selected_indices)
            collate = collate_fn_improved_mtl
            model = ImprovedShallowLSTM(len(selected_indices), hidden_size, len(PREDICTION_TARGETS), dropout).to(DEVICE)
        else:
            train_ds = YoYSequenceDatasetMTL(mtl_train_data, max_seq_length, scaler)
            val_ds = YoYSequenceDatasetMTL(mtl_val_data, max_seq_length, scaler)
            collate = None
            model = ShallowLSTMClassifier(len(feature_cols), hidden_size, len(PREDICTION_TARGETS), dropout).to(DEVICE)

        if collate:
            train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate)
            val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=collate)
        else:
            train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
            val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

        _, best_metric = train_and_eval(
            model, train_loader, val_loader, optimizer,
            is_improved=is_improved, trial=trial, loss_type=loss_type, huber_delta=huber_delta
        )
        return best_metric
    return objective

def evaluate_real_metrics(model, loader, device):
    model.eval()

    all_preds = []
    all_targets = []
    all_masks = []

    with torch.no_grad():

        for batch in loader:

            bx, by, bmask, lengths = batch

            preds = model(
                bx.to(device),
                lengths
            )

            all_preds.append(preds.cpu().numpy())
            all_targets.append(by.numpy())
            all_masks.append(bmask.numpy())

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_targets)
    mask = np.vstack(all_masks).astype(bool)

    # back to real scale
    y_pred = np.sign(y_pred) * np.expm1(np.abs(y_pred))
    y_true = np.sign(y_true) * np.expm1(np.abs(y_true))

    errors = np.abs(y_pred - y_true)
    squared_errors = (y_pred - y_true) ** 2

    mae = errors[mask].mean()
    rmse = np.sqrt(squared_errors[mask].mean())

    print(f"Test MAE  (Real Scale): {mae:.4f}")
    print(f"Test RMSE (Real Scale): {rmse:.4f}")

    return float(mae), float(rmse)

# Training

## Baseline MTL

In [ ]:
print("Baseline MTL (Comparing Losses)")
best_mtl_baselines = {}
for l_type in ['l1', 'mse', 'huber']:
    print(f"\nEvaluating Baseline with '{l_type}' Loss...")
    pruner = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=15)
    study = optuna.create_study(direction="minimize", study_name=f"mtl_base_{l_type}", pruner=pruner)
    study.optimize(get_hpo_objective(False, loss_type=l_type), n_trials=OPTUNA_TRIALS)
    best_mtl_baselines[l_type] = study.best_trial.params
    print(f"[{l_type.upper()} - MTL Base] Best Avg Val Metric (minimize): {study.best_value:.4f}")
    print(f" Best Params: {study.best_trial.params}")

[I 2026-05-27 23:36:53,208] A new study created in memory with name: mtl_base_l1


Baseline MTL (Comparing Losses)

Evaluating Baseline with 'l1' Loss...


[I 2026-05-27 23:37:21,708] Trial 0 finished with value: 7.708547115325928 and parameters: {'hidden_size': 64, 'lr': 0.0024561830181530947, 'batch_size': 16, 'weight_decay': 4.073563111630223e-05, 'dropout': 0.4552946184128708}. Best is trial 0 with value: 7.708547115325928.
[I 2026-05-27 23:37:41,707] Trial 1 finished with value: 7.718625545501709 and parameters: {'hidden_size': 64, 'lr': 0.0005348168313228728, 'batch_size': 16, 'weight_decay': 4.724536784754057e-05, 'dropout': 0.29887133584388825}. Best is trial 0 with value: 7.708547115325928.
[I 2026-05-27 23:38:23,947] Trial 2 finished with value: 7.726640224456787 and parameters: {'hidden_size': 32, 'lr': 0.0002523990382394761, 'batch_size': 64, 'weight_decay': 4.171485706338313e-06, 'dropout': 0.2109400209257967}. Best is trial 0 with value: 7.708547115325928.
[I 2026-05-27 23:39:06,785] Trial 3 finished with value: 7.716990947723389 and parameters: {'hidden_size': 128, 'lr': 0.00011145998046238179, 'batch_size': 16, 'weight_dec

[L1 - MTL Base] Best Avg Val Metric (minimize): 7.6932
 Best Params: {'hidden_size': 128, 'lr': 0.0014436947840858378, 'batch_size': 64, 'weight_decay': 3.876080827850374e-05, 'dropout': 0.39717688152505026}

Evaluating Baseline with 'mse' Loss...


[I 2026-05-28 00:06:30,368] Trial 0 finished with value: 7.724384307861328 and parameters: {'hidden_size': 128, 'lr': 0.0015145330934124405, 'batch_size': 16, 'weight_decay': 3.253113999087654e-05, 'dropout': 0.3269529567088313}. Best is trial 0 with value: 7.724384307861328.
[I 2026-05-28 00:06:38,304] Trial 1 finished with value: 7.764683246612549 and parameters: {'hidden_size': 64, 'lr': 0.004633281831020155, 'batch_size': 64, 'weight_decay': 1.4118533763819458e-06, 'dropout': 0.4763039504945981}. Best is trial 0 with value: 7.724384307861328.
[I 2026-05-28 00:07:26,753] Trial 2 finished with value: 7.72674560546875 and parameters: {'hidden_size': 32, 'lr': 0.0001222175136019422, 'batch_size': 16, 'weight_decay': 4.435005835871002e-06, 'dropout': 0.28090996968121984}. Best is trial 0 with value: 7.724384307861328.
[I 2026-05-28 00:07:41,832] Trial 3 finished with value: 7.844062328338623 and parameters: {'hidden_size': 32, 'lr': 0.0004939079126984807, 'batch_size': 64, 'weight_decay

[MSE - MTL Base] Best Avg Val Metric (minimize): 7.6997
 Best Params: {'hidden_size': 64, 'lr': 0.005358904243667092, 'batch_size': 16, 'weight_decay': 0.0007934779438993115, 'dropout': 0.46860618654311137}

Evaluating Baseline with 'huber' Loss...


[I 2026-05-28 00:31:50,971] Trial 0 finished with value: 7.700599670410156 and parameters: {'hidden_size': 32, 'lr': 0.008519254385214498, 'batch_size': 32, 'weight_decay': 5.473570001987502e-05, 'dropout': 0.43932997420898057, 'huber_delta': 1.7527364200689597}. Best is trial 0 with value: 7.700599670410156.
[I 2026-05-28 00:32:02,606] Trial 1 finished with value: 7.717978000640869 and parameters: {'hidden_size': 32, 'lr': 0.0024189438816534024, 'batch_size': 32, 'weight_decay': 1.776158101357255e-06, 'dropout': 0.46728196361073787, 'huber_delta': 1.5287744090532973}. Best is trial 0 with value: 7.700599670410156.
[I 2026-05-28 00:32:14,647] Trial 2 finished with value: 7.722008228302002 and parameters: {'hidden_size': 32, 'lr': 0.0011005108223455413, 'batch_size': 64, 'weight_decay': 0.0006224052693756531, 'dropout': 0.34777809003451254, 'huber_delta': 1.3008192146298994}. Best is trial 0 with value: 7.700599670410156.
[I 2026-05-28 00:32:27,533] Trial 3 finished with value: 7.702604

[HUBER - MTL Base] Best Avg Val Metric (minimize): 7.6960
 Best Params: {'hidden_size': 128, 'lr': 0.003673569654223351, 'batch_size': 16, 'weight_decay': 0.00021943085674158504, 'dropout': 0.3440568739399561, 'huber_delta': 1.4380843861038661}


## Improved MTL

In [ ]:
print("Improved MTL (Comparing Losses)")
best_mtl_imp = {}
for l_type in ['l1', 'mse', 'huber']:
    print(f"\nEvaluating Improved with '{l_type}' Loss...")
    pruner = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=15)
    study = optuna.create_study(direction="minimize", study_name=f"mtl_imp_{l_type}", pruner=pruner)
    study.optimize(get_hpo_objective(True, loss_type=l_type), n_trials=OPTUNA_TRIALS)
    best_mtl_imp[l_type] = study.best_trial.params
    print(f"[{l_type.upper()} - MTL Imp] Best Avg Val Metric (minimize): {study.best_value:.4f}")
    print(f" Best Params: {study.best_trial.params}")

[I 2026-05-28 00:57:20,876] A new study created in memory with name: mtl_imp_l1


Improved MTL (Comparing Losses)

Evaluating Improved with 'l1' Loss...


[I 2026-05-28 00:57:58,362] Trial 0 finished with value: 7.71516752243042 and parameters: {'hidden_size': 32, 'lr': 0.004511643533921293, 'batch_size': 16, 'weight_decay': 8.6585951257992e-06, 'dropout': 0.22385233887278816}. Best is trial 0 with value: 7.71516752243042.
[I 2026-05-28 00:58:51,539] Trial 1 finished with value: 7.7181243896484375 and parameters: {'hidden_size': 64, 'lr': 0.00017536326793130677, 'batch_size': 64, 'weight_decay': 2.0939538402178845e-05, 'dropout': 0.4171503485702226}. Best is trial 0 with value: 7.71516752243042.
[I 2026-05-28 00:59:29,181] Trial 2 finished with value: 7.717634677886963 and parameters: {'hidden_size': 128, 'lr': 0.00017921994721543407, 'batch_size': 64, 'weight_decay': 0.0001801048113560405, 'dropout': 0.2889659728898096}. Best is trial 0 with value: 7.71516752243042.
[I 2026-05-28 00:59:54,687] Trial 3 finished with value: 7.701570987701416 and parameters: {'hidden_size': 64, 'lr': 0.001681667100098799, 'batch_size': 32, 'weight_decay': 

[L1 - MTL Imp] Best Avg Val Metric (minimize): 7.6943
 Best Params: {'hidden_size': 32, 'lr': 0.003401507766189891, 'batch_size': 16, 'weight_decay': 7.839968960863435e-06, 'dropout': 0.2723634682240227}

Evaluating Improved with 'mse' Loss...


[I 2026-05-28 01:39:49,247] Trial 0 finished with value: 7.732018947601318 and parameters: {'hidden_size': 32, 'lr': 0.005352079410973245, 'batch_size': 32, 'weight_decay': 2.6071846106065405e-06, 'dropout': 0.27946559844786134}. Best is trial 0 with value: 7.732018947601318.
[I 2026-05-28 01:40:04,485] Trial 1 finished with value: 7.706556797027588 and parameters: {'hidden_size': 128, 'lr': 0.001219385767745531, 'batch_size': 64, 'weight_decay': 0.00016489777912956463, 'dropout': 0.2954822446600905}. Best is trial 1 with value: 7.706556797027588.
[I 2026-05-28 01:40:26,558] Trial 2 finished with value: 7.736567974090576 and parameters: {'hidden_size': 32, 'lr': 0.004193703789948248, 'batch_size': 32, 'weight_decay': 2.448344319703648e-05, 'dropout': 0.2794239544549575}. Best is trial 1 with value: 7.706556797027588.
[I 2026-05-28 01:41:53,507] Trial 3 finished with value: 7.817846775054932 and parameters: {'hidden_size': 32, 'lr': 0.00012715774086285704, 'batch_size': 64, 'weight_deca

[MSE - MTL Imp] Best Avg Val Metric (minimize): 7.6911
 Best Params: {'hidden_size': 128, 'lr': 0.00873193948799961, 'batch_size': 32, 'weight_decay': 0.000431472809664785, 'dropout': 0.4108777207863841}

Evaluating Improved with 'huber' Loss...


[I 2026-05-28 02:13:06,920] Trial 0 finished with value: 7.733912944793701 and parameters: {'hidden_size': 64, 'lr': 0.0007050345327095489, 'batch_size': 64, 'weight_decay': 9.420521364236271e-05, 'dropout': 0.4651764901679453, 'huber_delta': 0.9518220664728315}. Best is trial 0 with value: 7.733912944793701.
[I 2026-05-28 02:13:29,563] Trial 1 finished with value: 7.701531887054443 and parameters: {'hidden_size': 32, 'lr': 0.009649876940648899, 'batch_size': 32, 'weight_decay': 0.00047195056727470785, 'dropout': 0.2887697959253624, 'huber_delta': 1.6075713926931687}. Best is trial 1 with value: 7.701531887054443.
[I 2026-05-28 02:13:41,125] Trial 2 finished with value: 7.70311164855957 and parameters: {'hidden_size': 64, 'lr': 0.006256921721550774, 'batch_size': 64, 'weight_decay': 1.0383536946577804e-05, 'dropout': 0.3649447577492552, 'huber_delta': 1.5578675126668808}. Best is trial 1 with value: 7.701531887054443.
[I 2026-05-28 02:14:14,254] Trial 3 finished with value: 7.745927333

[HUBER - MTL Imp] Best Avg Val Metric (minimize): 7.6881
 Best Params: {'hidden_size': 128, 'lr': 0.005559384700410929, 'batch_size': 32, 'weight_decay': 0.000994493139509451, 'dropout': 0.30855438256245166, 'huber_delta': 0.8493557973953794}


In [ ]:
print("\n--- Baseline MTL Best Params ---")
for l_type, params in best_mtl_baselines.items():
    print(f"[Baseline - {l_type.upper()}] \n {params}\n")

print("\n--- Improved MTL Best Params ---")
for l_type, params in best_mtl_imp.items():
    print(f"[Improved - {l_type.upper()}] \n {params}\n")


--- Baseline MTL Best Params ---
[Baseline - L1] 
 {'hidden_size': 128, 'lr': 0.0014436947840858378, 'batch_size': 64, 'weight_decay': 3.876080827850374e-05, 'dropout': 0.39717688152505026}

[Baseline - MSE] 
 {'hidden_size': 64, 'lr': 0.005358904243667092, 'batch_size': 16, 'weight_decay': 0.0007934779438993115, 'dropout': 0.46860618654311137}

[Baseline - HUBER] 
 {'hidden_size': 128, 'lr': 0.003673569654223351, 'batch_size': 16, 'weight_decay': 0.00021943085674158504, 'dropout': 0.3440568739399561, 'huber_delta': 1.4380843861038661}


--- Improved MTL Best Params ---
[Improved - L1] 
 {'hidden_size': 32, 'lr': 0.003401507766189891, 'batch_size': 16, 'weight_decay': 7.839968960863435e-06, 'dropout': 0.2723634682240227}

[Improved - MSE] 
 {'hidden_size': 128, 'lr': 0.00873193948799961, 'batch_size': 32, 'weight_decay': 0.000431472809664785, 'dropout': 0.4108777207863841}

[Improved - HUBER] 
 {'hidden_size': 128, 'lr': 0.005559384700410929, 'batch_size': 32, 'weight_decay': 0.000994

In [ ]:
def build_loaders_and_model(is_improved, params):
    dropout_val = params.get("dropout", 0.2)
    if is_improved:
        train_ds = ImprovedYoYDatasetMTL(mtl_train_data_imp, robust_scaler, selected_indices)
        val_ds = ImprovedYoYDatasetMTL(mtl_val_data_imp, robust_scaler, selected_indices)
        test_ds = ImprovedYoYDatasetMTL(mtl_test_data_imp, robust_scaler, selected_indices)
        collate = collate_fn_improved_mtl
        model = ImprovedShallowLSTM(len(selected_indices), params["hidden_size"], len(PREDICTION_TARGETS), dropout_val).to(DEVICE)
    else:
        train_ds = YoYSequenceDatasetMTL(mtl_train_data, max_seq_length, scaler)
        val_ds = YoYSequenceDatasetMTL(mtl_val_data, max_seq_length, scaler)
        test_ds = YoYSequenceDatasetMTL(mtl_test_data, max_seq_length, scaler)
        collate = None
        model = ShallowLSTMClassifier(len(feature_cols), params["hidden_size"], len(PREDICTION_TARGETS), dropout_val).to(DEVICE)

    if collate:
        train_loader = DataLoader(train_ds, batch_size=params["batch_size"], shuffle=True, collate_fn=collate)
        val_loader = DataLoader(val_ds, batch_size=params["batch_size"], shuffle=False, collate_fn=collate)
        test_loader = DataLoader(test_ds, batch_size=params["batch_size"], shuffle=False, collate_fn=collate)
    else:
        train_loader = DataLoader(train_ds, batch_size=params["batch_size"], shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=params["batch_size"], shuffle=False)
        test_loader = DataLoader(test_ds, batch_size=params["batch_size"], shuffle=False)

    return model, train_loader, val_loader, test_loader


def evaluate_loader(model, loader, is_improved):
    model.eval()
    all_preds, all_targets, all_masks = [], [], []
    with torch.no_grad():
        for batch in loader:
            if is_improved:
                bx, by, bmask, lengths = batch
                logits = model(bx.to(DEVICE), lengths)
                all_masks.append(bmask.numpy())
            else:
                bx, by, bmask = batch
                logits = model(bx.to(DEVICE))
                all_masks.append(bmask.numpy())

            all_preds.append(logits.cpu().numpy())
            all_targets.append(by.numpy())

    if not all_preds:
        return None

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_targets)

    if is_improved:
        y_pred = np.sign(y_pred) * np.expm1(np.abs(y_pred))
        y_true = np.sign(y_true) * np.expm1(np.abs(y_true))

    mask = np.vstack(all_masks)
    res = {}
    for i, t in enumerate(PREDICTION_TARGETS):
        m = mask[:, i]
        if m.sum() == 0:
            continue
        res[t] = np.mean(np.abs(y_pred[m, i] - y_true[m, i]))
    return res


def final_train_and_test(is_improved, params, loss_type='l1'):
    model, train_loader, val_loader, test_loader = build_loaders_and_model(
        is_improved, params
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])
    huber_delta = params.get("huber_delta", 1.0)

    model, _ = train_and_eval(
        model, train_loader, val_loader, optimizer,
        is_improved=is_improved, loss_type=loss_type, huber_delta=huber_delta
    )

    return evaluate_loader(model, test_loader, is_improved=is_improved)

def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)

N_RUNS = 20
print(f"Computing Final Test Metrics using Test Sets across {N_RUNS} runs...")
final_results = []

def run_multiple_times(pipeline_name, is_improved, best_params_dict):
    for l_type, params in best_params_dict.items():
        print(f"Running {pipeline_name} with {l_type.upper()} loss...")
        target_results = {t: [] for t in PREDICTION_TARGETS}
        for run_idx in range(N_RUNS):
            set_seed(SEED + run_idx)
            res = final_train_and_test(is_improved, params, loss_type=l_type)
            for target, val in res.items():
                target_results[target].append(val)

        for target in PREDICTION_TARGETS:
            avg_mae = np.mean(target_results[target])
            std_mae = np.std(target_results[target])
            final_results.append({
                "Pipeline": pipeline_name,
                "Loss": l_type.upper(),
                "Target": target,
                "Test MAE (Mean)": avg_mae,
                "Deviation": std_mae
            })

Computing Final Test Metrics using Test Sets across 20 runs...


In [ ]:
final_results = []
run_multiple_times("Baseline", False, best_mtl_baselines)
run_multiple_times("Improved", True, best_mtl_imp)

df_final = pd.DataFrame(final_results)

# Create a pivot table for easy comparison
pivot_df = df_final.pivot_table(
    index=["Target", "Loss"],
    columns=["Pipeline"],
    values=["Test MAE (Mean)", "Deviation"]
)
display(pivot_df)

Running Baseline with L1 loss...
Running Baseline with MSE loss...
Running Baseline with HUBER loss...
Running Improved with L1 loss...
Running Improved with MSE loss...
Running Improved with HUBER loss...


Deviation           Test MAE (Mean)          
Pipeline          Baseline  Improved        Baseline  Improved
Target     Loss                                               
EBITDA     HUBER  0.040016  0.009347        1.537412  1.501781
           L1     0.018248  0.008191        1.497533  1.492321
           MSE    0.104793  0.036764        1.669935  1.533402
Net_Income HUBER  0.022528  0.008445        1.564425  1.553314
           L1     0.008348  0.008066        1.555753  1.561342
           MSE    0.038836  0.007787        1.594874  1.555342
ROA        HUBER  0.027862  0.012895        1.596761  1.587349
           L1     0.010313  0.012361        1.583823  1.587322
           MSE    0.060500  0.010091        1.647139  1.583727

## Segmented Performance Analysis

In [ ]:
def evaluate_group_pipeline(is_improved, best_params_dict):
    group_results = []

    for l_type, params in best_params_dict.items():
        # Store individual run MAEs here to compute standard deviation
        group_run_maes = {grp: {t: [] for t in PREDICTION_TARGETS} for grp in ticker_groups.keys()}

        for run_idx in range(N_RUNS):
            set_seed(SEED + run_idx)
            # Re-train model for this run
            model, train_loader, val_loader, _ = build_loaders_and_model(is_improved, params)
            optimizer = torch.optim.Adam(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])
            huber_delta = params.get("huber_delta", 1.0)
            model, _ = train_and_eval(model, train_loader, val_loader, optimizer, is_improved=is_improved, loss_type=l_type, huber_delta=huber_delta)

            for group_name, tickers in ticker_groups.items():
                if is_improved:
                    group_df = mtl_test_data_imp[mtl_test_data_imp['company'].isin(tickers)]
                    if group_df.empty: continue
                    group_ds = ImprovedYoYDatasetMTL(group_df, robust_scaler, selected_indices)
                    collate = collate_fn_improved_mtl
                else:
                    group_df = mtl_test_data[mtl_test_data['company'].isin(tickers)]
                    if group_df.empty: continue
                    group_ds = YoYSequenceDatasetMTL(group_df, max_seq_length, scaler)
                    collate = None

                loader = DataLoader(group_ds, batch_size=params["batch_size"], shuffle=False, collate_fn=collate)
                metrics = evaluate_loader(model, loader, is_improved=is_improved)

                if metrics:
                    for target, val in metrics.items():
                        group_run_maes[group_name][target].append(val)

        # Compute mean and std
        for group_name in ticker_groups.keys():
            for target in PREDICTION_TARGETS:
                if len(group_run_maes[group_name][target]) > 0:
                    maes = group_run_maes[group_name][target]
                    group_results.append({
                        "Loss": l_type.upper(),
                        "Group": group_name,
                        "Target": target,
                        "MAE (Mean)": np.mean(maes),
                        "Deviation": np.std(maes)
                    })

    return group_results

print("Evaluating Improved MTL across losses and groups...")
imp_group_stats = evaluate_group_pipeline(True, best_mtl_imp)

print("\nEvaluating Baseline MTL across losses and groups...")
base_group_stats = evaluate_group_pipeline(False, best_mtl_baselines)

df_imp_group = pd.DataFrame(imp_group_stats)
df_base_group = pd.DataFrame(base_group_stats)

print("\n--- Grouped MAE: Improved MTL ---")
if not df_imp_group.empty:
    display(df_imp_group.pivot_table(index=['Target', 'Loss'], columns='Group', values=['MAE (Mean)', 'Deviation']))

print("\n--- Grouped MAE: Baseline MTL ---")
if not df_base_group.empty:
    display(df_base_group.pivot_table(index=['Target', 'Loss'], columns='Group', values=['MAE (Mean)', 'Deviation']))

Evaluating Improved MTL across losses and groups...

Evaluating Baseline MTL across losses and groups...

--- Grouped MAE: Improved MTL ---


Deviation                     MAE (Mean)                    
Group                  MIB   Mid-cap Small-cap        MIB   Mid-cap Small-cap
Target     Loss                                                              
EBITDA     HUBER  0.013345  0.012285  0.011936   1.882129  0.432301  2.077805
           L1     0.007175  0.010151  0.011694   1.872868  0.423073  2.068846
           MSE    0.016179  0.029560  0.056993   1.879454  0.456574  2.127214
Net_Income HUBER  0.019447  0.015695  0.004571   0.727523  1.660671  1.794760
           L1     0.018631  0.018250  0.014093   0.728950  1.658343  1.811956
           MSE    0.020231  0.017087  0.003697   0.728444  1.674364  1.789339
ROA        HUBER  0.022916  0.024162  0.004770   0.724385  1.634140  1.882975
           L1     0.018667  0.024420  0.013050   0.722308  1.619287  1.893402
           MSE    0.018359  0.019872  0.005220   0.722233  1.631258  1.877700


--- Grouped MAE: Baseline MTL ---


Deviation                     MAE (Mean)                    
Group                  MIB   Mid-cap Small-cap        MIB   Mid-cap Small-cap
Target     Loss                                                              
EBITDA     HUBER  0.032670  0.048373  0.037989   1.897592  0.477673  2.114572
           L1     0.018431  0.021678  0.020876   1.868880  0.431836  2.074885
           MSE    0.089618  0.128185  0.095976   2.017731  0.637593  2.234647
Net_Income HUBER  0.041453  0.044896  0.011992   0.733051  1.674605  1.806381
           L1     0.022736  0.014829  0.010894   0.722157  1.649563  1.809530
           MSE    0.070602  0.070106  0.020684   0.782812  1.724882  1.816690
ROA        HUBER  0.046131  0.051967  0.015194   0.734569  1.644067  1.891570
           L1     0.021650  0.019594  0.012843   0.719380  1.612774  1.892010
           MSE    0.093082  0.097450  0.028914   0.814015  1.724964  1.910691

### Directional Accuracy

In [ ]:
def calculate_directional_accuracy(model, loader, is_improved):
    model.eval()
    all_preds, all_targets, all_masks = [], [], []
    with torch.no_grad():
        for batch in loader:
            if is_improved:
                bx, by, bmask, lengths = batch
                logits = model(bx.to(DEVICE), lengths)
                all_masks.append(bmask.numpy())
            else:
                bx, by, bmask = batch
                logits = model(bx.to(DEVICE))
                all_masks.append(bmask.numpy())
            all_preds.append(logits.cpu().numpy())
            all_targets.append(by.numpy())

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_targets)
    mask = np.vstack(all_masks)

    accuracy_res = {}
    for i, t in enumerate(PREDICTION_TARGETS):
        m = mask[:, i]
        if m.sum() == 0: continue
        # Directional agreement: both positive or both negative
        correct_dir = (np.sign(y_pred[m, i]) == np.sign(y_true[m, i])).sum()
        accuracy_res[t] = correct_dir / m.sum()
    return accuracy_res

print("--- Directional Accuracy (Improved Pipeline) ---")
for l_type, params in best_mtl_imp.items():
    model_imp, _, _, test_loader_imp = build_loaders_and_model(True, params)
    dir_acc = calculate_directional_accuracy(model_imp, test_loader_imp, True)
    print(f"[{l_type.upper()}] Directional Accuracy:", dir_acc)

--- Directional Accuracy (Improved Pipeline) ---
[L1] Directional Accuracy: {'EBITDA': np.float64(0.4315352697095436), 'Net_Income': np.float64(0.5318275154004107), 'ROA': np.float64(0.4948665297741273)}
[MSE] Directional Accuracy: {'EBITDA': np.float64(0.6307053941908713), 'Net_Income': np.float64(0.4722792607802875), 'ROA': np.float64(0.5297741273100616)}
[HUBER] Directional Accuracy: {'EBITDA': np.float64(0.37136929460580914), 'Net_Income': np.float64(0.5503080082135524), 'ROA': np.float64(0.5174537987679672)}


### Specialized Models by Market Cap

In [ ]:
def run_dedicated_pipeline(group_name, tickers, loss_type='l1'):
    print(f"--- Dedicated Pipeline: {group_name} ({loss_type.upper()}) ---")
    group_df = improved_data_df[improved_data_df['company'].isin(tickers)].copy()
    if group_df.empty:
        print(f"No data for {group_name}")
        return

    g_train, g_val, g_test = split_by_year(group_df, TRAIN_YEAR_CUTOFF, VALID_YEAR_CUTOFF)

    # Scoping scaler to group
    g_scaler = RobustScaler()
    g_train_windows = np.vstack([row[:, selected_indices] for row in g_train["window_data"]])
    g_scaler.fit(g_train_windows)

    g_mtl_train = extract_mtl_df(g_train)
    g_mtl_val = extract_mtl_df(g_val)
    g_mtl_test = extract_mtl_df(g_test)

    def group_objective(trial):
        hidden_size = trial.suggest_categorical("hidden_size", [32, 64, 128])
        lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
        batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
        weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
        dropout = trial.suggest_float("dropout", 0.1, 0.5)

        train_ds = ImprovedYoYDatasetMTL(g_mtl_train, g_scaler, selected_indices)
        val_ds = ImprovedYoYDatasetMTL(g_mtl_val, g_scaler, selected_indices)

        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn_improved_mtl)
        val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn_improved_mtl)

        model = ImprovedShallowLSTM(len(selected_indices), hidden_size, len(PREDICTION_TARGETS), dropout).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

        _, best_metric = train_and_eval(model, train_loader, val_loader, optimizer, True, trial, loss_type=loss_type)
        return best_metric

    pruner = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=15)
    study = optuna.create_study(direction="minimize", study_name=f"mtl_imp_{group_name}", pruner=pruner)
    study.optimize(group_objective, n_trials=OPTUNA_TRIALS)

    best_params = study.best_trial.params
    print(f"Best Params for {group_name}: {best_params}\n")

    # Final evaluation multiple runs
    target_results = {t: [] for t in PREDICTION_TARGETS}

    for run_idx in range(N_RUNS):
        set_seed(SEED + run_idx)
        train_ds = ImprovedYoYDatasetMTL(g_mtl_train, g_scaler, selected_indices)
        val_ds = ImprovedYoYDatasetMTL(g_mtl_val, g_scaler, selected_indices)
        test_ds = ImprovedYoYDatasetMTL(g_mtl_test, g_scaler, selected_indices)

        train_loader = DataLoader(train_ds, batch_size=best_params["batch_size"], shuffle=True, collate_fn=collate_fn_improved_mtl)
        val_loader = DataLoader(val_ds, batch_size=best_params["batch_size"], shuffle=False, collate_fn=collate_fn_improved_mtl)
        test_loader = DataLoader(test_ds, batch_size=best_params["batch_size"], shuffle=False, collate_fn=collate_fn_improved_mtl)

        dropout_val = best_params.get("dropout", 0.2)
        model = ImprovedShallowLSTM(len(selected_indices), best_params["hidden_size"], len(PREDICTION_TARGETS), dropout_val).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=best_params["lr"], weight_decay=best_params["weight_decay"])

        model, _ = train_and_eval(model, train_loader, val_loader, optimizer, True, None, loss_type=loss_type)
        res = evaluate_loader(model, test_loader, True)

        if res:
            for target, val in res.items():
                target_results[target].append(val)

        group_stats = []

    # MAE per target
    for target in PREDICTION_TARGETS:
        avg_mae = np.mean(target_results[target])
        std_mae = np.std(target_results[target])

        group_stats.append({
            "Group": group_name,
            "Target": target,
            "Dedicated Test MAE": avg_mae,
            "Dedicated Deviation": std_mae
        })

    total_mae = np.mean([
        np.mean(target_results[target])
        for target in PREDICTION_TARGETS
    ])

    total_std = np.std([
        np.mean(target_results[target])
        for target in PREDICTION_TARGETS
    ])

    print(f"{group_name} - Total Test MAE: {total_mae:.4f} ± {total_std:.4f}")

    group_stats.append({
        "Group": group_name,
        "Target": "TOTAL",
        "Dedicated Test MAE": total_mae,
        "Dedicated Deviation": total_std
    })

    return group_stats

dedicated_all_stats = []
for group_name, tickers in ticker_groups.items():
    stats = run_dedicated_pipeline(group_name, tickers, loss_type='l1') # Defaulting to L1 as champion
    if stats:
        dedicated_all_stats.extend(stats)

df_dedicated = pd.DataFrame(dedicated_all_stats)
display(df_dedicated.pivot(index='Target', columns='Group', values='Dedicated Test MAE'))

--- Dedicated Pipeline: MIB (L1) ---


[I 2026-06-09 13:36:37,825] A new study created in memory with name: mtl_imp_MIB
[I 2026-06-09 13:37:01,540] Trial 0 finished with value: 3.2393970489501953 and parameters: {'hidden_size': 64, 'lr': 0.00033394693981808214, 'batch_size': 16, 'weight_decay': 5.520454939337981e-06, 'dropout': 0.1597438568222771}. Best is trial 0 with value: 3.2393970489501953.
[I 2026-06-09 13:37:10,768] Trial 1 finished with value: 3.2112629413604736 and parameters: {'hidden_size': 32, 'lr': 0.0034624333747282606, 'batch_size': 32, 'weight_decay': 0.00069538361166545, 'dropout': 0.31359549926897357}. Best is trial 1 with value: 3.2112629413604736.
[I 2026-06-09 13:37:15,528] Trial 2 finished with value: 3.2050540447235107 and parameters: {'hidden_size': 128, 'lr': 0.001090209309503447, 'batch_size': 16, 'weight_decay': 3.0235842334655288e-05, 'dropout': 0.11609442500888463}. Best is trial 2 with value: 3.2050540447235107.
[I 2026-06-09 13:37:24,722] Trial 3 finished with value: 3.1962947845458984 and par

Best Params for MIB: {'hidden_size': 64, 'lr': 0.0047639846588888755, 'batch_size': 32, 'weight_decay': 0.0004627785003174142, 'dropout': 0.137398162298415}

MIB - Total Test MAE: 1.1194 ± 0.5330
--- Dedicated Pipeline: Mid-cap (L1) ---


[I 2026-06-09 13:46:15,458] A new study created in memory with name: mtl_imp_Mid-cap
[I 2026-06-09 13:46:20,249] Trial 0 finished with value: 1.3854198455810547 and parameters: {'hidden_size': 64, 'lr': 0.0007760041563306649, 'batch_size': 32, 'weight_decay': 0.00032688538898296537, 'dropout': 0.11462011092617184}. Best is trial 0 with value: 1.3854198455810547.
[I 2026-06-09 13:46:25,663] Trial 1 finished with value: 1.3796652555465698 and parameters: {'hidden_size': 32, 'lr': 0.0023475976550181347, 'batch_size': 64, 'weight_decay': 0.0008037673215877992, 'dropout': 0.10280236130787435}. Best is trial 1 with value: 1.3796652555465698.
[I 2026-06-09 13:46:29,413] Trial 2 finished with value: 1.405164361000061 and parameters: {'hidden_size': 64, 'lr': 0.0005172517117526508, 'batch_size': 32, 'weight_decay': 5.21249595830484e-05, 'dropout': 0.10665703567990432}. Best is trial 1 with value: 1.3796652555465698.
[I 2026-06-09 13:46:34,169] Trial 3 finished with value: 1.3795342445373535 and

Best Params for Mid-cap: {'hidden_size': 64, 'lr': 0.005426748767587874, 'batch_size': 32, 'weight_decay': 3.6520300079748134e-06, 'dropout': 0.2908379909201838}

Mid-cap - Total Test MAE: 1.2394 ± 0.5697
--- Dedicated Pipeline: Small-cap (L1) ---


[I 2026-06-09 13:57:30,157] A new study created in memory with name: mtl_imp_Small-cap
[I 2026-06-09 13:57:42,166] Trial 0 finished with value: 13.257041931152344 and parameters: {'hidden_size': 64, 'lr': 0.005848022240010461, 'batch_size': 16, 'weight_decay': 3.736112590892339e-06, 'dropout': 0.3433044313153144}. Best is trial 0 with value: 13.257041931152344.
[I 2026-06-09 13:57:53,661] Trial 1 finished with value: 13.288570404052734 and parameters: {'hidden_size': 32, 'lr': 0.0015539476187347461, 'batch_size': 16, 'weight_decay': 4.5568320096635844e-05, 'dropout': 0.2718966340549591}. Best is trial 0 with value: 13.257041931152344.
[I 2026-06-09 13:58:07,763] Trial 2 finished with value: 13.258105278015137 and parameters: {'hidden_size': 32, 'lr': 0.00419146898779243, 'batch_size': 16, 'weight_decay': 2.272519836209071e-06, 'dropout': 0.1319286950823879}. Best is trial 0 with value: 13.257041931152344.
[I 2026-06-09 13:58:32,269] Trial 3 finished with value: 13.247562408447266 and p

Best Params for Small-cap: {'hidden_size': 128, 'lr': 0.0024073927504210404, 'batch_size': 16, 'weight_decay': 2.879599343783324e-05, 'dropout': 0.3793695200631374}

Small-cap - Total Test MAE: 1.9426 ± 0.1155


Group,MIB,Mid-cap,Small-cap
Target,,,
EBITDA,1.872972,0.434066,2.098847
Net_Income,0.724902,1.663558,1.823405
ROA,0.760404,1.620463,1.905453
TOTAL,1.119426,1.239362,1.942568


In [22]:
# ============================================================
# TRAIN FINAL DEDICATED MODEL FOR EACH MARKET-CAP GROUP
# ============================================================

GROUP_PARAMS = {
    "MIB": {
        "hidden_size": 64,
        "lr": 0.0047639846588888755,
        "batch_size": 32,
        "weight_decay": 0.0004627785003174142,
        "dropout": 0.137398162298415
    },
    "Mid-cap": {
        "hidden_size": 64,
        "lr": 0.005426748767587874,
        "batch_size": 32,
        "weight_decay": 3.6520300079748134e-06,
        "dropout": 0.2908379909201838
    },
    "Small-cap": {
        "hidden_size": 128,
        "lr": 0.0024073927504210404,
        "batch_size": 16,
        "weight_decay": 2.879599343783324e-05,
        "dropout": 0.3793695200631374
    }
}

results = []

for group_name, tickers in ticker_groups.items():

    print("\n" + "=" * 60)
    print(group_name)
    print("=" * 60)

    params = GROUP_PARAMS[group_name]

    set_seed(42)

    group_df = improved_data_df[
        improved_data_df["company"].isin(tickers)
    ].copy()

    g_train, g_val, g_test = split_by_year(
        group_df,
        TRAIN_YEAR_CUTOFF,
        VALID_YEAR_CUTOFF
    )

    scaler = RobustScaler()

    train_windows = np.vstack([
        row[:, selected_indices]
        for row in g_train["window_data"]
    ])

    scaler.fit(train_windows)

    train_ds = ImprovedYoYDatasetMTL(
        extract_mtl_df(g_train),
        scaler,
        selected_indices
    )

    val_ds = ImprovedYoYDatasetMTL(
        extract_mtl_df(g_val),
        scaler,
        selected_indices
    )

    test_ds = ImprovedYoYDatasetMTL(
        extract_mtl_df(g_test),
        scaler,
        selected_indices
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=params["batch_size"],
        shuffle=True,
        collate_fn=collate_fn_improved_mtl
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=params["batch_size"],
        shuffle=False,
        collate_fn=collate_fn_improved_mtl
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=params["batch_size"],
        shuffle=False,
        collate_fn=collate_fn_improved_mtl
    )

    model = ImprovedShallowLSTM(
        input_size=len(selected_indices),
        hidden_size=params["hidden_size"],
        num_targets=len(PREDICTION_TARGETS),
        dropout=params["dropout"]
    ).to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=params["lr"],
        weight_decay=params["weight_decay"]
    )

    model, _ = train_and_eval(
        model,
        train_loader,
        val_loader,
        optimizer,
        True,
        None,
        loss_type='l1'
    )

    print("\nTEST RESULTS")
    mae, rmse = evaluate_real_metrics(
        model,
        test_loader,
        DEVICE
    )

    n_samples = len(test_ds)

    results.append({
        "Group": group_name,
        "MAE": mae,
        "RMSE": rmse,
        "N": n_samples
    })

df_results = pd.DataFrame(results)

print("\n")
print("=" * 60)
print("FINAL RESULTS")
print("=" * 60)

print(df_results)

weighted_mae = (
    (df_results["MAE"] * df_results["N"]).sum()
    / df_results["N"].sum()
)

weighted_rmse = (
    (df_results["RMSE"] * df_results["N"]).sum()
    / df_results["N"].sum()
)

print("\nWeighted MAE:")
print(weighted_mae)

print("\nWeighted RMSE:")
print(weighted_rmse)


MIB

TEST RESULTS
Test MAE  (Real Scale): 1.1349
Test RMSE (Real Scale): 8.2930

Mid-cap

TEST RESULTS
Test MAE  (Real Scale): 1.2548
Test RMSE (Real Scale): 8.7773

Small-cap

TEST RESULTS
Test MAE  (Real Scale): 1.9577
Test RMSE (Real Scale): 8.6113


FINAL RESULTS
       Group       MAE      RMSE    N
0        MIB  1.134931  8.293011   87
1    Mid-cap  1.254799  8.777294  156
2  Small-cap  1.957686  8.611273  241

Weighted MAE:
1.5832434488722116

Weighted RMSE:
8.607575554493046
